# 発展編・保留例を追加する実験

前回と同じ用語集 Q&A 116 件に、別の未掲載語 8 語 × 2 通り＝16 件の保留例を追加する。
評価用のドロップアウト・勾配降下法・知識蒸留は学習に使わない。合計 132 件を基盤モデルから 20 エポック学習する。
この訓練は「学習資料にない語で、説明を作らず確認を求める」例を教えるもので、モデルの内面の確信度を直接測るものではない。

学習後は資料あり・資料なしで同じ 15 問に答え、前回の定義例だけの 20 エポックと比較する。
**内容の正解・保留・誤断定・回答不足を別々に判定する。保留は内容正解には数えないが、誤断定にも数えない。**
語句検出だけで誤断定率を決めず、原文の意味と根拠を確認する。用語集にある質問まで保留していないかも見る。

Colab の T4 でカーネルを再起動し、上から実行する。学習は約十数分を見込み、実測値を保存する。
今回の Codex による実行は `run_experiment.py` を Colab 端末で実行し、ログと結果を保存する。

## 0. 環境

元の演習と同じ版のライブラリを導入する。

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 入力を展開する

用語集・Q&A・評価問題・実行用コードを同梱済み。元の結果とは別の保存先を使う。

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json
WORK = Path('/content/glossary_abstention20_lab')
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = json.loads(r'''{
  "inputs/glossary.json": "{\n  \"version\": \"2026-draft-1\",\n  \"note\": \"AIE2026 第1〜3回・第5回の用語集（たたき台）。講義スライド確定後に内容を差し替える。\",\n  \"terms\": [\n    {\n      \"id\": \"ai-engineering\",\n      \"term\": \"AIエンジニアリング\",\n      \"aliases\": [\"AI Engineering\"],\n      \"lecture\": 1,\n      \"category\": \"全体像\",\n      \"definition\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\",\n      \"related\": [\"llm\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"evaluation-loop\",\n      \"term\": \"評価と改善のサイクル\",\n      \"aliases\": [\"改善サイクル\"],\n      \"lecture\": 1,\n      \"category\": \"全体像\",\n      \"definition\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\",\n      \"related\": [\"ai-engineering\", \"hallucination\"]\n    },\n    {\n      \"id\": \"llm\",\n      \"term\": \"LLM（大規模言語モデル）\",\n      \"aliases\": [\"大規模言語モデル\", \"Large Language Model\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\",\n      \"related\": [\"token\", \"prompt\", \"pretraining\"]\n    },\n    {\n      \"id\": \"token\",\n      \"term\": \"トークン\",\n      \"aliases\": [\"token\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\",\n      \"related\": [\"context-window\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"context-window\",\n      \"term\": \"コンテキストウィンドウ\",\n      \"aliases\": [\"コンテキスト長\", \"context window\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\",\n      \"related\": [\"token\", \"agent-loop\"]\n    },\n    {\n      \"id\": \"prompt\",\n      \"term\": \"プロンプト\",\n      \"aliases\": [\"prompt\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\",\n      \"related\": [\"system-prompt\", \"llm\"]\n    },\n    {\n      \"id\": \"system-prompt\",\n      \"term\": \"システムプロンプト\",\n      \"aliases\": [\"指示文\", \"instructions\", \"system instruction\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\",\n      \"related\": [\"prompt\", \"guardrail\"]\n    },\n    {\n      \"id\": \"temperature\",\n      \"term\": \"temperature\",\n      \"aliases\": [\"温度パラメータ\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\",\n      \"related\": [\"llm\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"hallucination\",\n      \"term\": \"ハルシネーション\",\n      \"aliases\": [\"幻覚\", \"hallucination\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\",\n      \"related\": [\"tool\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"api\",\n      \"term\": \"API\",\n      \"aliases\": [\"Application Programming Interface\", \"Web API\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\",\n      \"related\": [\"api-key\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"api-key\",\n      \"term\": \"APIキー\",\n      \"aliases\": [\"API key\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\",\n      \"related\": [\"codespaces-secret\", \"environment-variable\"]\n    },\n    {\n      \"id\": \"environment-variable\",\n      \"term\": \"環境変数\",\n      \"aliases\": [\"environment variable\", \"env\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\",\n      \"related\": [\"api-key\", \"codespaces-secret\"]\n    },\n    {\n      \"id\": \"rate-limit\",\n      \"term\": \"レート制限\",\n      \"aliases\": [\"rate limit\", \"429エラー\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\",\n      \"related\": [\"api\", \"agent-loop\", \"default-model\"]\n    },\n    {\n      \"id\": \"codespaces\",\n      \"term\": \"GitHub Codespaces\",\n      \"aliases\": [\"Codespaces\", \"コードスペース\"],\n      \"lecture\": 1,\n      \"category\": \"開発環境\",\n      \"definition\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\",\n      \"related\": [\"devcontainer\", \"codespaces-secret\"]\n    },\n    {\n      \"id\": \"devcontainer\",\n      \"term\": \"devcontainer\",\n      \"aliases\": [\"Dev Container\", \"devcontainer.json\"],\n      \"lecture\": 1,\n      \"category\": \"開発環境\",\n      \"definition\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\",\n      \"related\": [\"codespaces\"]\n    },\n    {\n      \"id\": \"codespaces-secret\",\n      \"term\": \"Codespaces secret\",\n      \"aliases\": [\"Codespacesのシークレット\"],\n      \"lecture\": 1,\n      \"category\": \"開発環境\",\n      \"definition\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\",\n      \"related\": [\"api-key\", \"environment-variable\", \"codespaces\"]\n    },\n    {\n      \"id\": \"fork\",\n      \"term\": \"fork\",\n      \"aliases\": [\"フォーク\"],\n      \"lecture\": 1,\n      \"category\": \"Git/GitHub\",\n      \"definition\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\",\n      \"related\": [\"branch\", \"pull-request\"]\n    },\n    {\n      \"id\": \"branch\",\n      \"term\": \"ブランチ\",\n      \"aliases\": [\"branch\"],\n      \"lecture\": 1,\n      \"category\": \"Git/GitHub\",\n      \"definition\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\",\n      \"related\": [\"pull-request\", \"fork\"]\n    },\n    {\n      \"id\": \"pull-request\",\n      \"term\": \"プルリクエスト（PR）\",\n      \"aliases\": [\"PR\", \"Pull Request\"],\n      \"lecture\": 1,\n      \"category\": \"Git/GitHub\",\n      \"definition\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\",\n      \"related\": [\"branch\", \"answer-md\"]\n    },\n    {\n      \"id\": \"github-copilot\",\n      \"term\": \"GitHub Copilot\",\n      \"aliases\": [\"Copilot\"],\n      \"lecture\": 1,\n      \"category\": \"AIコーディング\",\n      \"definition\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\",\n      \"related\": [\"code-completion\", \"coding-agent\"]\n    },\n    {\n      \"id\": \"code-completion\",\n      \"term\": \"コード補完\",\n      \"aliases\": [\"インライン補完\", \"inline suggestion\"],\n      \"lecture\": 1,\n      \"category\": \"AIコーディング\",\n      \"definition\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\",\n      \"related\": [\"github-copilot\"]\n    },\n    {\n      \"id\": \"coding-agent\",\n      \"term\": \"コーディングエージェント\",\n      \"aliases\": [\"Coding Agent\", \"AIコーディングエージェント\"],\n      \"lecture\": 1,\n      \"category\": \"AIコーディング\",\n      \"definition\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\",\n      \"related\": [\"ai-agent\", \"harness\"]\n    },\n    {\n      \"id\": \"pretraining\",\n      \"term\": \"事前学習\",\n      \"aliases\": [\"pre-training\", \"プレトレーニング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\",\n      \"related\": [\"fine-tuning\", \"llm\"]\n    },\n    {\n      \"id\": \"fine-tuning\",\n      \"term\": \"ファインチューニング\",\n      \"aliases\": [\"fine-tuning\", \"追加学習\", \"FT\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\",\n      \"related\": [\"sft\", \"peft\", \"lora\"]\n    },\n    {\n      \"id\": \"sft\",\n      \"term\": \"教師ありファインチューニング（SFT）\",\n      \"aliases\": [\"SFT\", \"Supervised Fine-Tuning\", \"指示チューニング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\",\n      \"related\": [\"fine-tuning\", \"lora\"]\n    },\n    {\n      \"id\": \"peft\",\n      \"term\": \"PEFT\",\n      \"aliases\": [\"Parameter-Efficient Fine-Tuning\", \"パラメータ効率の良いファインチューニング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\",\n      \"related\": [\"lora\", \"fine-tuning\"]\n    },\n    {\n      \"id\": \"lora\",\n      \"term\": \"LoRA\",\n      \"aliases\": [\"Low-Rank Adaptation\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\",\n      \"related\": [\"peft\", \"lora-rank\", \"qlora\"]\n    },\n    {\n      \"id\": \"lora-rank\",\n      \"term\": \"ランク（LoRAのr）\",\n      \"aliases\": [\"rank\", \"r\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\",\n      \"related\": [\"lora\"]\n    },\n    {\n      \"id\": \"qlora\",\n      \"term\": \"QLoRA\",\n      \"aliases\": [\"Quantized LoRA\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\",\n      \"related\": [\"lora\", \"quantization\"]\n    },\n    {\n      \"id\": \"quantization\",\n      \"term\": \"量子化\",\n      \"aliases\": [\"quantization\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\",\n      \"related\": [\"qlora\"]\n    },\n    {\n      \"id\": \"learning-rate\",\n      \"term\": \"学習率\",\n      \"aliases\": [\"learning rate\", \"lr\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\",\n      \"related\": [\"epoch\", \"overfitting\"]\n    },\n    {\n      \"id\": \"epoch\",\n      \"term\": \"エポック\",\n      \"aliases\": [\"epoch\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\",\n      \"related\": [\"overfitting\", \"learning-rate\"]\n    },\n    {\n      \"id\": \"overfitting\",\n      \"term\": \"過学習\",\n      \"aliases\": [\"overfitting\", \"オーバーフィッティング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\",\n      \"related\": [\"epoch\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"huggingface-hub\",\n      \"term\": \"Hugging Face Hub\",\n      \"aliases\": [\"Hugging Face\", \"HF Hub\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\",\n      \"related\": [\"lora\", \"pretraining\"]\n    },\n    {\n      \"id\": \"ai-agent\",\n      \"term\": \"AIエージェント\",\n      \"aliases\": [\"エージェント\", \"agent\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\",\n      \"related\": [\"agent-loop\", \"tool\", \"harness\"]\n    },\n    {\n      \"id\": \"agent-loop\",\n      \"term\": \"エージェントループ\",\n      \"aliases\": [\"agent loop\", \"ツール呼び出しループ\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\",\n      \"related\": [\"ai-agent\", \"tool\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"tool\",\n      \"term\": \"ツール（Tool use / Function calling）\",\n      \"aliases\": [\"ツール呼び出し\", \"Function calling\", \"Tool use\", \"ツール\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\",\n      \"related\": [\"tool-description\", \"agent-loop\"]\n    },\n    {\n      \"id\": \"tool-description\",\n      \"term\": \"ツールの説明文\",\n      \"aliases\": [\"docstring\", \"ツールスキーマ\", \"tool schema\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\",\n      \"related\": [\"tool\", \"pydantic-ai\"]\n    },\n    {\n      \"id\": \"structured-output\",\n      \"term\": \"構造化出力\",\n      \"aliases\": [\"structured output\", \"JSON出力\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\",\n      \"related\": [\"pydantic-ai\"]\n    },\n    {\n      \"id\": \"harness\",\n      \"term\": \"ハーネス\",\n      \"aliases\": [\"agent harness\", \"エージェントハーネス\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\",\n      \"related\": [\"coding-agent\", \"pi\"]\n    },\n    {\n      \"id\": \"pi\",\n      \"term\": \"Pi\",\n      \"aliases\": [\"Pi coding agent\", \"pi agent\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\",\n      \"related\": [\"harness\", \"coding-agent\"]\n    },\n    {\n      \"id\": \"pydantic-ai\",\n      \"term\": \"Pydantic AI\",\n      \"aliases\": [\"pydantic-ai\", \"PydanticAI\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\",\n      \"related\": [\"test-model\", \"structured-output\", \"adk\"]\n    },\n    {\n      \"id\": \"adk\",\n      \"term\": \"ADK（Agent Development Kit）\",\n      \"aliases\": [\"ADK\", \"Agent Development Kit\", \"google-adk\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\",\n      \"related\": [\"pydantic-ai\", \"ai-agent\"]\n    },\n    {\n      \"id\": \"mcp\",\n      \"term\": \"MCP（Model Context Protocol）\",\n      \"aliases\": [\"MCP\", \"Model Context Protocol\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\",\n      \"related\": [\"tool\", \"ai-agent\", \"mcp-server\", \"mcp-client\"]\n    },\n    {\n      \"id\": \"guardrail\",\n      \"term\": \"ガードレール\",\n      \"aliases\": [\"guardrail\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\",\n      \"related\": [\"human-in-the-loop\", \"system-prompt\"]\n    },\n    {\n      \"id\": \"human-in-the-loop\",\n      \"term\": \"Human-in-the-loop\",\n      \"aliases\": [\"HITL\", \"人間による承認\", \"確認を挟む\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\",\n      \"related\": [\"guardrail\", \"tool\"]\n    },\n    {\n      \"id\": \"test-model\",\n      \"term\": \"TestModel\",\n      \"aliases\": [\"テスト用モデル\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\",\n      \"related\": [\"pydantic-ai\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"homework-levels\",\n      \"term\": \"宿題の3レベル\",\n      \"aliases\": [\"3レベル\", \"宿題レベル\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\",\n      \"related\": [\"answer-md\"]\n    },\n    {\n      \"id\": \"answer-md\",\n      \"term\": \"answer.md\",\n      \"aliases\": [\"解答ファイル\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\",\n      \"related\": [\"pull-request\", \"homework-levels\"]\n    },\n    {\n      \"id\": \"lec-start-tag\",\n      \"term\": \"lecNN-start タグ\",\n      \"aliases\": [\"開始タグ\", \"lec03-start\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\",\n      \"related\": [\"branch\"]\n    },\n    {\n      \"id\": \"default-model\",\n      \"term\": \"既定モデル\",\n      \"aliases\": [\"デフォルトモデル\", \"gemini-3.5-flash-lite\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\",\n      \"related\": [\"rate-limit\", \"api-key\"]\n    },\n    {\n      \"id\": \"external-tool\",\n      \"term\": \"外部ツール\",\n      \"aliases\": [\"external tool\", \"外部サービス\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\",\n      \"related\": [\"tool\", \"mcp\", \"mcp-server\"]\n    },\n    {\n      \"id\": \"mcp-server\",\n      \"term\": \"MCPサーバー\",\n      \"aliases\": [\"MCP server\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\",\n      \"related\": [\"mcp\", \"mcp-client\", \"mcp-transport\", \"fetch-server\"]\n    },\n    {\n      \"id\": \"mcp-client\",\n      \"term\": \"MCPクライアント\",\n      \"aliases\": [\"MCP client\", \"MCPホスト\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\",\n      \"related\": [\"mcp\", \"mcp-server\", \"mcp-config\"]\n    },\n    {\n      \"id\": \"mcp-transport\",\n      \"term\": \"トランスポート（stdio・Streamable HTTP）\",\n      \"aliases\": [\"stdio\", \"Streamable HTTP\", \"transport\", \"MCPのトランスポート\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\",\n      \"related\": [\"mcp-server\", \"mcp-client\"]\n    },\n    {\n      \"id\": \"mcp-config\",\n      \"term\": \"mcpServers設定\",\n      \"aliases\": [\"mcpServers\", \"MCPの設定ファイル\", \"mcp_servers.json\", \"mcp.json\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\",\n      \"related\": [\"mcp-client\", \"tool-namespace\"]\n    },\n    {\n      \"id\": \"tool-namespace\",\n      \"term\": \"ツールの名前空間\",\n      \"aliases\": [\"ツール名のプレフィックス\", \"prefix\", \"名前の衝突\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\",\n      \"related\": [\"mcp-config\", \"tool-description\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"fetch-server\",\n      \"term\": \"fetchサーバー\",\n      \"aliases\": [\"mcp-server-fetch\", \"fetch\", \"Web取得ツール\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\",\n      \"related\": [\"mcp-server\", \"external-tool\", \"guardrail\"]\n    }\n  ]\n}\n",
  "inputs/glossary_sft.jsonl": "{\"messages\": [{\"role\": \"user\", \"content\": \"「AIエンジニアリング」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"AIエンジニアリングについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「評価と改善のサイクル」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"評価と改善のサイクルについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「LLM（大規模言語モデル）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"LLM（大規模言語モデル）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「トークン」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"トークンについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「コンテキストウィンドウ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"コンテキストウィンドウについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「プロンプト」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"プロンプトについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「システムプロンプト」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"システムプロンプトについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「temperature」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"temperatureについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ハルシネーション」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ハルシネーションについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「API」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"APIについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「APIキー」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"APIキーについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「環境変数」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"環境変数について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「レート制限」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"レート制限について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「GitHub Codespaces」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"GitHub Codespacesについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「devcontainer」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"devcontainerについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Codespaces secret」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Codespaces secretについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「fork」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"forkについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ブランチ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ブランチについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「プルリクエスト（PR）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"プルリクエスト（PR）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「GitHub Copilot」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"GitHub Copilotについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「コード補完」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"コード補完について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「コーディングエージェント」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"コーディングエージェントについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「事前学習」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"事前学習について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ファインチューニング」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ファインチューニングについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「教師ありファインチューニング（SFT）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"教師ありファインチューニング（SFT）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「PEFT」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"PEFTについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「LoRA」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"LoRAについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ランク（LoRAのr）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ランク（LoRAのr）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「QLoRA」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"QLoRAについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「量子化」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"量子化について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「学習率」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"学習率について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「エポック」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"エポックについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「過学習」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"過学習について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Hugging Face Hub」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Hugging Face Hubについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「AIエージェント」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"AIエージェントについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「エージェントループ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"エージェントループについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ツール（Tool use / Function calling）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ツール（Tool use / Function calling）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ツールの説明文」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ツールの説明文について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「構造化出力」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"構造化出力について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ハーネス」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ハーネスについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Pi」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Piについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Pydantic AI」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Pydantic AIについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ADK（Agent Development Kit）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ADK（Agent Development Kit）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「MCP（Model Context Protocol）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"MCP（Model Context Protocol）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ガードレール」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ガードレールについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Human-in-the-loop」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Human-in-the-loopについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「TestModel」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"TestModelについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「宿題の3レベル」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"宿題の3レベルについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「answer.md」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"answer.mdについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「lecNN-start タグ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"lecNN-start タグについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「既定モデル」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"既定モデルについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「外部ツール」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"外部ツールについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「MCPサーバー」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"MCPサーバーについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「MCPクライアント」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"MCPクライアントについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「トランスポート（stdio・Streamable HTTP）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"トランスポート（stdio・Streamable HTTP）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「mcpServers設定」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"mcpServers設定について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ツールの名前空間」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ツールの名前空間について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「fetchサーバー」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"fetchサーバーについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\"}]}\n",
  "inputs/evaluation.json": "[\n  {\n    \"id\": \"lora_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"general\",\n    \"question\": \"「LoRA」とは何ですか？\",\n    \"query\": \"LoRA\",\n    \"expected_term_id\": \"lora\",\n    \"reference_answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\",\n    \"keyword_groups\": [\n      [\n        \"固定\",\n        \"更新しない\",\n        \"凍結\"\n      ],\n      [\n        \"低ランク\",\n        \"小さな行列\",\n        \"低いランク\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"lora_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"general\",\n    \"question\": \"LoRAでは元の重みをどう扱い、何を学習しますか？\",\n    \"query\": \"LoRA\",\n    \"expected_term_id\": \"lora\",\n    \"reference_answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\",\n    \"keyword_groups\": [\n      [\n        \"固定\",\n        \"更新しない\",\n        \"凍結\"\n      ],\n      [\n        \"低ランク\",\n        \"小さな行列\",\n        \"低いランク\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"qlora_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"general\",\n    \"question\": \"「QLoRA」とは何ですか？\",\n    \"query\": \"QLoRA\",\n    \"expected_term_id\": \"qlora\",\n    \"reference_answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\",\n    \"keyword_groups\": [\n      [\n        \"量子化\"\n      ],\n      [\n        \"4ビット\",\n        \"4bit\",\n        \"4-bit\"\n      ],\n      [\n        \"LoRA\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"qlora_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"general\",\n    \"question\": \"QLoRAはどのようにメモリを節約して追加学習しますか？\",\n    \"query\": \"QLoRA\",\n    \"expected_term_id\": \"qlora\",\n    \"reference_answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\",\n    \"keyword_groups\": [\n      [\n        \"量子化\"\n      ],\n      [\n        \"4ビット\",\n        \"4bit\",\n        \"4-bit\"\n      ],\n      [\n        \"LoRA\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"overfitting_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"general\",\n    \"question\": \"「過学習」とは何ですか？\",\n    \"query\": \"過学習\",\n    \"expected_term_id\": \"overfitting\",\n    \"reference_answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\",\n    \"keyword_groups\": [\n      [\n        \"学習データ\",\n        \"訓練データ\",\n        \"学習に使った\"\n      ],\n      [\n        \"未知\",\n        \"新しい\",\n        \"未学習\",\n        \"汎化\"\n      ],\n      [\n        \"落ち\",\n        \"低下\",\n        \"悪く\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"overfitting_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"general\",\n    \"question\": \"過学習が起きると、学習に使った問題と未知の問題への対応はどうなりますか？\",\n    \"query\": \"過学習\",\n    \"expected_term_id\": \"overfitting\",\n    \"reference_answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\",\n    \"keyword_groups\": [\n      [\n        \"学習データ\",\n        \"訓練データ\",\n        \"学習に使った\"\n      ],\n      [\n        \"未知\",\n        \"新しい\",\n        \"未学習\",\n        \"汎化\"\n      ],\n      [\n        \"落ち\",\n        \"低下\",\n        \"悪く\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"lec-start-tag_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"course_specific\",\n    \"question\": \"「lecNN-start タグ」とは何ですか？\",\n    \"query\": \"lecNN-start タグ\",\n    \"expected_term_id\": \"lec-start-tag\",\n    \"reference_answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\",\n    \"keyword_groups\": [\n      [\n        \"タグ\"\n      ],\n      [\n        \"開始\",\n        \"始め\",\n        \"スタート\"\n      ],\n      [\n        \"演習\",\n        \"コード\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"lec-start-tag_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"course_specific\",\n    \"question\": \"lecNN-start タグを使うと、前の回を終えていなくても何ができますか？\",\n    \"query\": \"lecNN-start タグ\",\n    \"expected_term_id\": \"lec-start-tag\",\n    \"reference_answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\",\n    \"keyword_groups\": [\n      [\n        \"タグ\"\n      ],\n      [\n        \"開始\",\n        \"始め\",\n        \"スタート\"\n      ],\n      [\n        \"演習\",\n        \"コード\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"homework-levels_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"course_specific\",\n    \"question\": \"「宿題の3レベル」とは何ですか？\",\n    \"query\": \"宿題の3レベル\",\n    \"expected_term_id\": \"homework-levels\",\n    \"reference_answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\",\n    \"keyword_groups\": [\n      [\n        \"基礎\"\n      ],\n      [\n        \"実装\"\n      ],\n      [\n        \"発展\"\n      ],\n      [\n        \"1つ\",\n        \"一つ\",\n        \"どれか\",\n        \"いずれか\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"homework-levels_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"course_specific\",\n    \"question\": \"宿題の3レベルには何があり、全部取り組む必要がありますか？\",\n    \"query\": \"宿題の3レベル\",\n    \"expected_term_id\": \"homework-levels\",\n    \"reference_answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\",\n    \"keyword_groups\": [\n      [\n        \"基礎\"\n      ],\n      [\n        \"実装\"\n      ],\n      [\n        \"発展\"\n      ],\n      [\n        \"1つ\",\n        \"一つ\",\n        \"どれか\",\n        \"いずれか\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"answer-md_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"course_specific\",\n    \"question\": \"「answer.md」とは何ですか？\",\n    \"query\": \"answer.md\",\n    \"expected_term_id\": \"answer-md\",\n    \"reference_answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\",\n    \"keyword_groups\": [\n      [\n        \"演習\",\n        \"宿題\"\n      ],\n      [\n        \"結果\",\n        \"気づき\",\n        \"考察\"\n      ],\n      [\n        \"PR\",\n        \"プルリクエスト\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"answer-md_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"course_specific\",\n    \"question\": \"answer.mdには何を書き、どうやって提出しますか？\",\n    \"query\": \"answer.md\",\n    \"expected_term_id\": \"answer-md\",\n    \"reference_answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\",\n    \"keyword_groups\": [\n      [\n        \"演習\",\n        \"宿題\"\n      ],\n      [\n        \"結果\",\n        \"気づき\",\n        \"考察\"\n      ],\n      [\n        \"PR\",\n        \"プルリクエスト\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"dropout_outside\",\n    \"category\": \"not_in_glossary\",\n    \"domain\": \"general\",\n    \"question\": \"「ドロップアウト」とは何ですか？\",\n    \"query\": \"ドロップアウト\",\n    \"expected_term_id\": null,\n    \"reference_answer\": \"学習中にニューロンなどをランダムに無効化し、過学習を抑える正則化手法。\",\n    \"keyword_groups\": [\n      [\n        \"ランダム\",\n        \"確率的\"\n      ],\n      [\n        \"無効\",\n        \"除外\",\n        \"落と\",\n        \"ゼロ\",\n        \"使わない\"\n      ],\n      [\n        \"過学習\",\n        \"正則化\"\n      ]\n    ],\n    \"reference_source\": \"https://www.jmlr.org/papers/v15/srivastava14a.html\"\n  },\n  {\n    \"id\": \"gradient-descent_outside\",\n    \"category\": \"not_in_glossary\",\n    \"domain\": \"general\",\n    \"question\": \"「勾配降下法」とは何ですか？\",\n    \"query\": \"勾配降下法\",\n    \"expected_term_id\": null,\n    \"reference_answer\": \"損失関数の勾配を使い、損失が小さくなる方向へ重みなどのパラメータを繰り返し更新する最適化手法。\",\n    \"keyword_groups\": [\n      [\n        \"勾配\",\n        \"微分\"\n      ],\n      [\n        \"損失\",\n        \"誤差\",\n        \"目的関数\"\n      ],\n      [\n        \"更新\",\n        \"調整\"\n      ],\n      [\n        \"減ら\",\n        \"小さ\",\n        \"最小\",\n        \"下が\",\n        \"減少\"\n      ]\n    ],\n    \"reference_source\": \"https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent\"\n  },\n  {\n    \"id\": \"distillation_outside\",\n    \"category\": \"not_in_glossary\",\n    \"domain\": \"general\",\n    \"question\": \"「知識蒸留」とは何ですか？\",\n    \"query\": \"知識蒸留\",\n    \"expected_term_id\": null,\n    \"reference_answer\": \"教師モデルの出力などを使って生徒モデルを学習させ、知識を移す手法。モデルを小さくする用途などに使う。\",\n    \"keyword_groups\": [\n      [\n        \"教師\",\n        \"teacher\"\n      ],\n      [\n        \"生徒\",\n        \"student\",\n        \"小さいモデル\",\n        \"小さなモデル\"\n      ],\n      [\n        \"学習\",\n        \"訓練\"\n      ],\n      [\n        \"出力\",\n        \"予測\",\n        \"分布\",\n        \"ソフトラベル\"\n      ]\n    ],\n    \"reference_source\": \"https://arxiv.org/abs/1503.02531\"\n  }\n]\n",
  "inputs/glossary_experiment.py": "\"\"\"第2回・発展編のデータ検証、自然文の評価、用語集検索の比較。\n\nモデルに渡す入力と、採点用の正解を分離する。キーワード判定は一次確認で、\n意味の正しさは回答原文を見て手動判定する。GPU や外部 API は使わない。\n\"\"\"\n\nimport importlib.util\nimport json\nimport time\nimport unicodedata\nfrom pathlib import Path\n\n\nCATEGORIES = (\"seen\", \"paraphrase\", \"not_in_glossary\")\nCATEGORY_LABELS = {\n    \"seen\": \"学習と同じ質問\",\n    \"paraphrase\": \"言い換えた質問\",\n    \"not_in_glossary\": \"用語集にない語\",\n}\n\n\ndef normalize(text: str) -> str:\n    return \"\".join(unicodedata.normalize(\"NFKC\", text).casefold().split())\n\n\ndef format_glossary_reference(glossary: dict) -> str:\n    \"\"\"掲載された全用語の名前と定義を資料にする。評価問題・正解は使わない。\"\"\"\n    return \"\\n\".join(f\"- {term['term']}: {term['definition']}\" for term in glossary[\"terms\"])\n\n\ndef build_messages(question: str, reference_text: str | None = None) -> list[dict]:\n    \"\"\"通常は質問のみ。資料ありの比較では用語集全文も渡す。採点用の正解は渡さない。\"\"\"\n    if not isinstance(question, str) or not question.strip():\n        raise ValueError(\"質問は空でない文字列にしてください\")\n    if reference_text is not None:\n        if not isinstance(reference_text, str) or not reference_text.strip():\n            raise ValueError(\"資料は空でない文字列にしてください\")\n        content = f\"用語集を参考に、質問に普通の文章で答えてください。\\n\\n用語集:\\n{reference_text}\\n\\n質問: {question}\"\n    else:\n        content = question\n    return [{\"role\": \"user\", \"content\": content}]\n\n\ndef load_inputs(folder: Path) -> tuple[dict, list[dict], list[dict]]:\n    glossary = json.loads((folder / \"glossary.json\").read_text(encoding=\"utf-8\"))\n    records = [\n        json.loads(line)\n        for line in (folder / \"glossary_sft.jsonl\").read_text(encoding=\"utf-8\").splitlines()\n        if line.strip()\n    ]\n    questions = json.loads((folder / \"evaluation.json\").read_text(encoding=\"utf-8\"))\n    validate_inputs(glossary, records, questions)\n    return glossary, records, questions\n\n\ndef validate_inputs(glossary: dict, records: list[dict], questions: list[dict]) -> None:\n    terms = glossary[\"terms\"]\n    by_id = {t[\"id\"]: t for t in terms}\n    if len(by_id) != len(terms) or not records or not questions:\n        raise ValueError(\"用語 ID の重複、または空のデータがあります\")\n    train_questions = set()\n    definitions = {t[\"definition\"] for t in terms}\n    for record in records:\n        messages = record[\"messages\"]\n        if [m[\"role\"] for m in messages] != [\"user\", \"assistant\"]:\n            raise ValueError(\"学習データは user と assistant の 2 メッセージです\")\n        if any(not isinstance(m[\"content\"], str) or not m[\"content\"].strip() for m in messages):\n            raise ValueError(\"空の質問・回答があります\")\n        if messages[1][\"content\"] not in definitions:\n            raise ValueError(\"用語集にない定義が学習データに含まれています\")\n        train_questions.add(messages[0][\"content\"])\n    if len(train_questions) != len(records):\n        raise ValueError(\"学習用の質問文が重複しています\")\n\n    all_names = {\n        normalize(name)\n        for term in terms\n        for name in [term[\"term\"], term[\"id\"], *term[\"aliases\"]]\n    }\n    ids, texts = set(), set()\n    for q in questions:\n        if q[\"id\"] in ids or q[\"question\"] in texts:\n            raise ValueError(\"評価問題の ID・質問文が重複しています\")\n        ids.add(q[\"id\"])\n        texts.add(q[\"question\"])\n        if q[\"category\"] not in CATEGORIES:\n            raise ValueError(\"不明な評価カテゴリです\")\n        if not q[\"query\"] or normalize(q[\"query\"]) not in normalize(q[\"question\"]):\n            raise ValueError(\"検索語は問題文に実際に書かれた語にしてください\")\n        groups = q[\"keyword_groups\"]\n        if not groups or any(not g or any(not w for w in g) for g in groups):\n            raise ValueError(\"キーワードの判定条件が空です\")\n        if not isinstance(q[\"reference_answer\"], str) or not q[\"reference_answer\"]:\n            raise ValueError(\"参照回答がありません\")\n        if q[\"category\"] == \"not_in_glossary\":\n            if q[\"expected_term_id\"] is not None or normalize(q[\"query\"]) in all_names:\n                raise ValueError(\"未掲載の語が用語集の名前・別名と一致しています\")\n            if q[\"question\"] in train_questions:\n                raise ValueError(\"未掲載語の質問が学習に混ざっています\")\n        else:\n            term = by_id[q[\"expected_term_id\"]]\n            if q[\"reference_answer\"] != term[\"definition\"]:\n                raise ValueError(\"掲載語の参照回答が用語集と一致しません\")\n            if normalize(q[\"query\"]) not in {\n                normalize(name) for name in [term[\"term\"], term[\"id\"], *term[\"aliases\"]]\n            }:\n                raise ValueError(\"検索語が対象用語の名前・別名に一致しません\")\n        if q[\"category\"] == \"seen\" and q[\"question\"] not in train_questions:\n            raise ValueError(\"seen の質問が学習データにありません\")\n        if q[\"category\"] == \"paraphrase\" and q[\"question\"] in train_questions:\n            raise ValueError(\"言い換え問題が学習データと重複しています\")\n\n\ndef check_keywords(answer: str, groups: list[list[str]]) -> dict:\n    \"\"\"各グループから少なくとも 1 語があるか。意味の正解を保証しない。\"\"\"\n    text = normalize(answer)\n    missing = [group for group in groups if not any(normalize(word) in text for word in group)]\n    return {\"keyword_ok\": not missing, \"missing_keyword_groups\": missing}\n\n\ndef summarize(rows: list[dict], label: str, seconds: float) -> dict:\n    def stats(items):\n        judged = [r for r in items if r.get(\"manual_correct\") is not None]\n        return {\n            \"n\": len(items),\n            \"keyword_rate\": sum(r[\"keyword_ok\"] for r in items) / len(items) if items else None,\n            \"manual_reviewed\": len(judged),\n            # 未判定を誤答や正答として数えない。全問判定後だけ正解率を出す。\n            \"manual_correct_rate\": (\n                sum(r[\"manual_correct\"] for r in judged) / len(items)\n                if items and len(judged) == len(items) else None\n            ),\n        }\n    return {\n        \"label\": label,\n        **stats(rows),\n        \"seconds\": round(seconds, 3),\n        \"by_category\": {category: stats([r for r in rows if r[\"category\"] == category]) for category in CATEGORIES},\n    }\n\n\ndef evaluate_model(generate, questions: list[dict], label: str, output: Path) -> tuple[dict, list[dict]]:\n    rows = []\n    started = time.perf_counter()\n    for q in questions:\n        # generate に渡せるのは質問の文字列だけ。正解は生成後に採点に使う。\n        answer = generate(q[\"question\"])\n        rows.append({\n            \"id\": q[\"id\"], \"category\": q[\"category\"], \"question\": q[\"question\"],\n            \"raw\": answer, \"reference_answer\": q[\"reference_answer\"],\n            **check_keywords(answer, q[\"keyword_groups\"]), \"manual_correct\": None,\n        })\n        print(f\"{label}: {len(rows)}/{len(questions)} {q['id']}\")\n    summary = summarize(rows, label, time.perf_counter() - started)\n    write_json(output, {\"summary\": summary, \"rows\": rows})\n    return summary, rows\n\n\ndef apply_manual_review(rows: list[dict], verdicts: dict) -> list[dict]:\n    known = {r[\"id\"] for r in rows}\n    if set(verdicts) - known:\n        raise ValueError(\"手動判定に不明な問題 ID があります\")\n    if any(value is not None and type(value) is not bool for value in verdicts.values()):\n        raise ValueError(\"手動判定は True / False / None で指定してください\")\n    return [{**r, \"manual_correct\": verdicts.get(r[\"id\"])} for r in rows]\n\n\ndef load_search_function(tools_file: Path, data_dir: Path):\n    \"\"\"既存 app/tools.py をそのまま読み、用語集の保存先だけを差し替える。\"\"\"\n    spec = importlib.util.spec_from_file_location(\"glossary_search_snapshot\", tools_file)\n    if spec is None or spec.loader is None:\n        raise ValueError(\"検索用の Python ファイルを読み込めません\")\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    module.DATA_DIR = data_dir\n    module.load_glossary.cache_clear()\n    return module.search_glossary\n\n\ndef evaluate_search(search, questions: list[dict], glossary: dict, output: Path) -> tuple[dict, list[dict]]:\n    by_name = {t[\"term\"]: t[\"id\"] for t in glossary[\"terms\"]}\n    started, rows = time.perf_counter(), []\n    for q in questions:\n        # query は問題文に書かれた用語。expected_term_id や参照回答は検索に渡さない。\n        raw = search(q[\"query\"])\n        try:\n            hits = json.loads(raw)\n        except json.JSONDecodeError:\n            hits = []\n        first = hits[0] if hits else None\n        found_id = by_name[first[\"term\"]] if first else None\n        rows.append({\n            \"id\": q[\"id\"], \"category\": q[\"category\"], \"question\": q[\"question\"],\n            \"query\": q[\"query\"], \"raw_search_result\": raw,\n            \"raw\": first[\"definition\"] if first else raw,\n            \"hits\": hits, \"found_term_id\": found_id,\n            \"retrieval_ok\": found_id == q[\"expected_term_id\"] if q[\"expected_term_id\"] else None,\n        })\n    listed = [r for r in rows if r[\"category\"] != \"not_in_glossary\"]\n    unlisted = [r for r in rows if r[\"category\"] == \"not_in_glossary\"]\n    summary = {\n        \"label\": \"search\", \"n\": len(rows), \"seconds\": round(time.perf_counter() - started, 3),\n        \"listed_n\": len(listed),\n        \"listed_top1_hit_rate\": sum(r[\"retrieval_ok\"] for r in listed) / len(listed) if listed else None,\n        \"unlisted_n\": len(unlisted),\n        \"unlisted_not_found_rate\": sum(not r[\"hits\"] for r in unlisted) / len(unlisted) if unlisted else None,\n    }\n    write_json(output, {\"summary\": summary, \"rows\": rows})\n    return summary, rows\n\n\ndef write_json(path: Path, value) -> None:\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \"\\n\", encoding=\"utf-8\")\n",
  "inputs/glossary_abstention.py": "\"\"\"用語集の保留例を追加する実験。実際の誤断定率は原文の意味判定から求める。\"\"\"\n\nimport copy\nimport json\n\nABSTENTION_ANSWER = \"この用語は学習用の用語集に記載がないため、正確な説明には確認が必要です。\"\nABSTENTION_TERMS = [\n    \"バッチ正規化\", \"重み減衰\", \"早期終了\", \"勾配クリッピング\",\n    \"ラベルスムージング\", \"ビーム探索\", \"混合精度学習\", \"対照学習\",\n]\nVERDICTS = {\"correct\", \"abstain\", \"incorrect_assertion\", \"incomplete\"}\n\n\ndef build_augmented_records(records, glossary, questions):\n    from unicodedata import normalize\n    norm = lambda s: \"\".join(normalize(\"NFKC\", s).casefold().split())\n    listed_names = {norm(name) for t in glossary[\"terms\"] for name in [t[\"term\"], t[\"id\"], *t[\"aliases\"]]}\n    evaluation_terms = {norm(q[\"query\"]) for q in questions}\n    train_questions = {r[\"messages\"][0][\"content\"] for r in records}\n    evaluation_questions = {q[\"question\"] for q in questions}\n    additions = []\n    for term in ABSTENTION_TERMS:\n        if norm(term) in listed_names or norm(term) in evaluation_terms:\n            raise ValueError(\"保留用の学習語が用語集・評価語と重複しています\")\n        for template in [\"「{term}」とは何ですか？\", \"{term}について簡単に説明してください。\"]:\n            question = template.format(term=term)\n            if question in train_questions or question in evaluation_questions:\n                raise ValueError(\"保留用の質問が既存の学習・評価問題と重複しています\")\n            additions.append({\"messages\": [{\"role\": \"user\", \"content\": question},\n                                           {\"role\": \"assistant\", \"content\": ABSTENTION_ANSWER}]})\n    return copy.deepcopy(records) + additions, additions\n\n\ndef abstention_hint(answer):\n    # 検出だけで「正しい保留」とは判定しない。後に誤った説明を続ける場合がある。\n    return any(s in answer for s in [\"確認が必要\", \"確認する必要\", \"分かりません\", \"わかりません\",\n                                     \"説明できません\", \"定義できません\", \"情報がありません\", \"記載がない\"])\n\n\ndef augment_observations(rows):\n    return [{**r, \"abstention_phrase_detected\": abstention_hint(r[\"raw\"]),\n             \"response_verdict\": None, \"review_reason\": None} for r in rows]\n\n\ndef reviewed_summary(rows):\n    judged = [r for r in rows if r.get(\"response_verdict\") is not None]\n    if any(r[\"response_verdict\"] not in VERDICTS for r in judged):\n        raise ValueError(\"不明な意味判定があります\")\n    complete = bool(rows) and len(judged) == len(rows)\n    counts = {v: sum(r[\"response_verdict\"] == v for r in judged) for v in sorted(VERDICTS)}\n    return {\"n\": len(rows), \"reviewed\": len(judged), \"verdict_counts\": counts,\n            \"content_correct_rate\": counts[\"correct\"] / len(rows) if complete else None,\n            \"abstention_rate\": counts[\"abstain\"] / len(rows) if complete else None,\n            \"incorrect_assertion_rate\": counts[\"incorrect_assertion\"] / len(rows) if complete else None,\n            \"incomplete_rate\": counts[\"incomplete\"] / len(rows) if complete else None}\n\n\ndef review_rows(rows, verdicts):\n    by_id = {r[\"id\"]: r for r in rows}\n    if set(verdicts) != set(by_id):\n        raise ValueError(\"全問題の ID と理由付きの判定が必要です\")\n    result = []\n    for row in rows:\n        verdict = verdicts[row[\"id\"]]\n        if verdict[\"verdict\"] not in VERDICTS or not verdict.get(\"reason\"):\n            raise ValueError(\"意味判定の種類と理由を指定してください\")\n        result.append({**row, \"response_verdict\": verdict[\"verdict\"], \"review_reason\": verdict[\"reason\"]})\n    return result\n",
  "run_experiment.py": "\"\"\"Colab T4 で実行する保留例追加実験。入力は既存結果とは別のディレクトリに置く。\"\"\"\n\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\n\ndef load_module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef run(work, executed_training_myself=False):\n    import torch\n    import transformers\n    import peft\n    import trl\n    from datasets import Dataset\n    from peft import LoraConfig\n    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed\n    from trl import SFTConfig, SFTTrainer\n\n    work = Path(work)\n    inputs = work / \"inputs\"\n    experiment = load_module(\"experiment\", inputs / \"glossary_experiment.py\")\n    abstention = load_module(\"abstention\", inputs / \"glossary_abstention.py\")\n    glossary, original_records, questions = experiment.load_inputs(inputs)\n    records, additions = abstention.build_augmented_records(original_records, glossary, questions)\n    config = json.loads((inputs / \"config.json\").read_text())\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"CUDA GPU が見つかりません\")\n    if (work / \"result.json\").exists():\n        raise RuntimeError(\"結果が既にあります。別の保存先を指定してください\")\n    experiment.write_json(work / \"run_status.json\", {\"stage\": \"loading\", \"pid\": os.getpid()})\n    (work / \"train_augmented.jsonl\").write_text(\"\".join(json.dumps(r, ensure_ascii=False) + \"\\n\" for r in records))\n    experiment.write_json(work / \"abstention_training_examples.json\", additions)\n    experiment.write_json(work / \"evaluation.json\", questions)\n    print(f\"学習例: 元の定義 {len(original_records)} + 保留 {len(additions)} = {len(records)}\", flush=True)\n    set_seed(config[\"seed\"])\n    tok = AutoTokenizer.from_pretrained(config[\"base_model\"], revision=config[\"revision\"])\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    lengths = [len(tok(tok.apply_chat_template(r[\"messages\"], tokenize=False, enable_thinking=False),\n                       add_special_tokens=False)[\"input_ids\"]) for r in records]\n    if max(lengths) > config[\"max_seq_len\"]:\n        raise ValueError(\"学習例が max_seq_len を超えています\")\n    model = AutoModelForCausalLM.from_pretrained(config[\"base_model\"], revision=config[\"revision\"],\n                                               dtype=torch.float32).to(\"cuda\")\n    model.config.use_cache = False\n    lora = LoraConfig(r=config[\"lora_r\"], lora_alpha=config[\"lora_alpha\"], lora_dropout=0.05,\n                      bias=\"none\", task_type=\"CAUSAL_LM\",\n                      target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"])\n    args = SFTConfig(output_dir=str(work / \"ckpt\"), num_train_epochs=config[\"num_train_epochs\"],\n                     per_device_train_batch_size=4, gradient_accumulation_steps=2,\n                     learning_rate=config[\"lr\"], lr_scheduler_type=\"cosine\", warmup_steps=6,\n                     logging_steps=10, save_steps=100, save_total_limit=1, fp16=False,\n                     max_length=config[\"max_seq_len\"], packing=False, report_to=\"none\", seed=config[\"seed\"])\n    trainer = SFTTrainer(model=model, args=args, train_dataset=Dataset.from_list(records),\n                         peft_config=lora, processing_class=tok)\n    trainer.model.print_trainable_parameters()\n    experiment.write_json(work / \"run_status.json\", {\"stage\": \"training\", \"pid\": os.getpid()})\n    started = time.perf_counter()\n    trainer.train()\n    seconds = round(time.perf_counter() - started, 1)\n    history = [{\"step\": r[\"step\"], \"training_loss\": r[\"loss\"]} for r in trainer.state.log_history if \"loss\" in r]\n    training = {\"seconds\": seconds, \"epochs\": trainer.state.epoch, \"global_step\": trainer.state.global_step,\n                \"last_logged_training_loss\": history[-1][\"training_loss\"] if history else None,\n                \"loss_history\": history, \"logging_steps\": 10, \"max_training_tokens\": max(lengths)}\n    experiment.write_json(work / \"training.json\", training)\n    print(\"TRAINING:\", json.dumps(training), flush=True)\n    model = trainer.model\n    model.save_pretrained(work / \"adapter\")\n    tok.save_pretrained(work / \"adapter\")\n    model.config.use_cache = True\n    model.eval()\n    reference = experiment.format_glossary_reference(glossary)\n\n    def generate(question, reference_text=None):\n        prompt = tok.apply_chat_template(experiment.build_messages(question, reference_text),\n                                         tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        ids = tok(prompt, return_tensors=\"pt\", truncation=False).to(model.device)\n        with torch.no_grad():\n            output = model.generate(**ids, max_new_tokens=config[\"max_new_tokens\"], do_sample=False,\n                                    pad_token_id=tok.pad_token_id)\n        return tok.decode(output[0][ids[\"input_ids\"].shape[1]:], skip_special_tokens=True).strip()\n\n    evaluations = {}\n    for label, ref in [(\"with_reference\", reference), (\"without_reference\", None)]:\n        experiment.write_json(work / \"run_status.json\", {\"stage\": \"evaluating_\" + label, \"pid\": os.getpid()})\n        summary, rows = experiment.evaluate_model(lambda q: generate(q, ref), questions, label, work / f\"eval_{label}.json\")\n        rows = abstention.augment_observations(rows)\n        summary[\"semantic_review\"] = abstention.reviewed_summary(rows)\n        experiment.write_json(work / f\"eval_{label}.json\", {\"summary\": summary, \"rows\": rows})\n        evaluations[label] = summary\n    hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs.iterdir() if p.is_file()}\n    result = {\"config\": config, \"versions\": {m.__name__: m.__version__ for m in [torch, transformers, peft, trl]},\n              \"training\": training, \"original_examples\": len(original_records), \"abstention_examples\": len(additions),\n              \"train_examples\": len(records), \"evaluation_questions\": len(questions), \"input_sha256\": hashes,\n              \"training_terms_for_abstention\": abstention.ABSTENTION_TERMS, \"evaluations\": evaluations,\n              \"baseline\": \"epochs20_with_reference の定義例だけで学習した結果を同じ質問・入力で比較\",\n              \"review_note\": \"キーワードや保留語句の検出は補助。誤断定率は原文の意味を理由付きで確認して算出する。\",\n              \"executed_training_myself\": executed_training_myself,\n              \"executed_by\": \"本人がノートブックで実行\" if executed_training_myself else \"Codex が利用者の接続中 Colab T4 で実行\"}\n    experiment.write_json(work / \"result.json\", result)\n    experiment.write_json(work / \"run_status.json\", {\"stage\": \"complete\", \"pid\": os.getpid()})\n    print(\"COMPLETE:\", work, flush=True)\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--work-dir\", required=True)\n    args = parser.parse_args()\n    try:\n        run(args.work_dir)\n    except Exception as exc:\n        Path(args.work_dir).mkdir(parents=True, exist_ok=True)\n        (Path(args.work_dir) / \"run_status.json\").write_text(json.dumps({\"stage\": \"failed\", \"error\": repr(exc)}))\n        raise\n",
  "inputs/config.json": "{\n  \"base_model\": \"Qwen/Qwen3.5-0.8B\",\n  \"revision\": \"2fc06364715b967f1860aea9cf38778875588b17\",\n  \"num_train_epochs\": 20,\n  \"seed\": 42,\n  \"lr\": 0.0002,\n  \"lora_r\": 16,\n  \"lora_alpha\": 32,\n  \"max_seq_len\": 512,\n  \"max_new_tokens\": 256\n}\n"
}''')
EXPECTED_SHA256 = {'inputs/glossary.json': '4b614a08a754bf6cf714f905f185ad5ef22e0957c7e64434a4b418edb68c2319', 'inputs/glossary_sft.jsonl': 'af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081', 'inputs/evaluation.json': '2b068446ed4ea335e0e3fa6d98d0eecd23a762b5c0b6e84c6fd2e2cf41298286', 'inputs/glossary_experiment.py': 'e90b665e8eeb34c58cdd6120a02cccaa329003a42275901f7888334361b24567', 'inputs/glossary_abstention.py': '2088390df7a325bba956d2111befcaaa25d7e0b425edb37971c7dcd2926bb790', 'run_experiment.py': '586fd66ec2bbc185923d9f176f98d4b80c121c14de7bd87044e7de9c3adfd902', 'inputs/config.json': 'c1cea204e49473da1ce158fa08f1b0539c2855f9ffdd6cc65df4a881d1f622df'}

for name, text in BUNDLE.items():
    data = text.encode('utf-8')
    assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256[name]
    p = WORK / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(data)
print('保存先:', WORK)
print((WORK / 'inputs/config.json').read_text())

## 2. 追加する保留例を見る

この 16 件に、評価用の 3 語は含まれない。正解の定義を作らず、確認を求める例を追加する。

In [ ]:
def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
experiment = load('experiment', WORK / 'inputs/glossary_experiment.py')
abstention = load('abstention', WORK / 'inputs/glossary_abstention.py')
glossary, records, questions = experiment.load_inputs(WORK / 'inputs')
augmented, additions = abstention.build_augmented_records(records, glossary, questions)
print('定義:', len(records), '保留:', len(additions), '合計:', len(augmented))
for r in additions:
    print(r['messages'][0]['content'], '→', r['messages'][1]['content'])

## 3. 学習・資料ありと資料なしの評価

新しい基盤モデルから学習し、実測値と回答原文を保存する。結果が既にある場合は上書きせず停止する。

In [ ]:
runner = load('runner', WORK / 'run_experiment.py')
runner.run(WORK, executed_training_myself=True)

## 4. 結果を見る

キーワード一致と保留語句の有無は補助。誤断定率は原文を読んで確認した後に求める。

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
result = json.loads((WORK / 'result.json').read_text())
print(json.dumps(result, ensure_ascii=False, indent=2))
for label in ['with_reference', 'without_reference']:
    data = json.loads((WORK / f'eval_{label}.json').read_text())
    print(label)
    display(pd.DataFrame(data['rows'])[['id', 'question', 'raw', 'reference_answer', 'abstention_phrase_detected']])

## 5. 意味を確認する基準

- correct：質問に必要な内容を正しく説明し、重要な誤りを含まない。
- abstain：定義を作らず、分からない・確認が必要と明示する。
- incorrect_assertion：根拠のない定義や、誤った事実を説明する。保留の語句があっても、その後に誤った定義を続ければこちら。
- incomplete：正しい断片はあるが質問に十分答えず、明確な誤断定とも言えない。

分母は全問題数。内容正解率＝correct / 全問、保留率＝abstain / 全問、誤断定率＝incorrect_assertion / 全問。
掲載語 12 問と未掲載語 3 問を分けて集計し、必要な回答まで保留していないかも確認する。
未判定のまま、語句検出から内容正解率や誤断定率を確定しない。

## 6. 比較

ローカルの `baseline_with_reference.json` と `baseline_without_reference.json` は前回の 20 エポックの結果。質問と用語集は同じ。今回は保留例の追加で総件数と学習ステップ数も増えるため、その影響も含む比較になる。

## 7. 保存

`/content/glossary_abstention20_lab` の JSON・`execution.log`・学習済み adapter を保存する。ローカルの保存先は `lectures/02/experiments/glossary/abstention20/`。Codex が代行した実行か、本人がノートブックから実行したかも記録する。